# FrozenLake LVR training on Colab A100

This notebook reproduces the custom FrozenLake training path for **Qwen2.5-VL-3B-Instruct** on one A100 40 GB.

The model sees only the initial board image and the game instruction. Intermediate successor-state images supervise latent tokens. The textual target contains only the navigation actions.

Safety gates are deliberate: environment validation and a one-batch forward pass must succeed before pilot or full training. Pilot, full training, validation, and final test are opt-in cells.

## Run order

1. Run **Configuration**, **Hardware check**, **Clone**, and **Install**.
2. The install cell restarts the runtime. After reconnection, continue at **Restore configuration**; do not rerun the install cell.
3. Download, verify, extract, validate, preflight, and smoke-test the dataset/model.
4. Enable the 10-step pilot only after the smoke test reports `status: ok`.
5. Enable full training only after the pilot completes successfully.

The full job writes to Google Drive by default for persistence. This is slower than local Colab storage, but survives runtime recycling.

In [ ]:
# @title 1. Configuration
from pathlib import Path
import json

REPO_URL = "https://github.com/Noridom1/lvr.git"  # @param {type:"string"}
REVISION = "main"  # @param {type:"string"}
DRIVE_URL = "https://drive.google.com/file/d/1XhmzxEM0HCn6CTnNVkR7cCQEHaXujZqA/view?usp=sharing"  # @param {type:"string"}
ARCHIVE_SHA256 = "D7B3C86A41F3CB1BE5468E9D7130E5F7568BC1CECB47DD93380A16392F705540"
EXPECTED_ARCHIVE_BYTES = 1842169309
MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"  # @param {type:"string"}
PREFER_FLASH_ATTENTION = True  # @param {type:"boolean"}

CONFIG_PATH = Path("/content/frozenlake_lvr_colab_config.json")
REPO_DIR = Path("/content/lvr")
ARCHIVE_PATH = Path("/content/frozenlake_training_samples.zip")
DATA_ROOT = REPO_DIR / "training_samples" / "frozenlake"

config = {
    "repo_url": REPO_URL,
    "revision": REVISION,
    "drive_url": DRIVE_URL,
    "archive_sha256": ARCHIVE_SHA256,
    "expected_archive_bytes": EXPECTED_ARCHIVE_BYTES,
    "model_name": MODEL_NAME,
    "prefer_flash_attention": PREFER_FLASH_ATTENTION,
}
CONFIG_PATH.write_text(json.dumps(config, indent=2), encoding="utf-8")
print(json.dumps(config, indent=2))
print(f"Configuration saved across runtime restart: {CONFIG_PATH}")

In [ ]:
# @title 2. Check the assigned Colab hardware
import shutil
import subprocess
import psutil
import torch

subprocess.run(["nvidia-smi"], check=True)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select an A100 GPU runtime before continuing.")

gpu_name = torch.cuda.get_device_name(0)
gpu_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
memory = psutil.virtual_memory()
disk = shutil.disk_usage("/content")

print(f"GPU: {gpu_name} ({gpu_gib:.1f} GiB)")
print(f"RAM: {memory.total / 1024**3:.1f} GiB total, {memory.available / 1024**3:.1f} GiB free")
print(f"Disk: {disk.free / 1024**3:.1f} GiB free")

if gpu_gib < 39:
    raise RuntimeError("This configuration expects approximately 40 GiB GPU VRAM.")
if memory.total / 1024**3 < 75:
    raise RuntimeError("ZeRO-3 CPU offload expects at least approximately 75 GiB system RAM.")
if disk.free / 1024**3 < 100:
    raise RuntimeError("At least approximately 100 GiB free local disk is recommended.")

In [ ]:
# @title 3. Clone the repository and resolve the requested revision
import subprocess
from pathlib import Path

if (REPO_DIR / ".git").is_dir():
    print(f"Reusing existing clone: {REPO_DIR}")
else:
    if REPO_DIR.exists() and any(REPO_DIR.iterdir()):
        raise RuntimeError(f"{REPO_DIR} exists and is not an empty Git checkout.")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
if REVISION == "main":
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", REVISION], check=True)

resolved_revision = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    text=True,
).strip()
print(f"Resolved revision: {resolved_revision}")

## Install boundary

The next cell installs the official PyTorch 2.6.0 CUDA 12.4 wheels, then only the dependencies imported by the FrozenLake path. It optionally builds FlashAttention and restarts the Colab runtime so Torch and compiled extensions load cleanly.

FlashAttention installation is allowed to fail: after restart, the notebook automatically falls back to PyTorch SDPA.

In [ ]:
# @title 4. Install FrozenLake dependencies and restart
import importlib.util
import os
import signal
import subprocess
import sys
from pathlib import Path

PYTORCH_INDEX = "https://download.pytorch.org/whl/cu124"
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "torch==2.6.0",
        "torchvision==0.21.0",
        "torchaudio==2.6.0",
        "--index-url",
        PYTORCH_INDEX,
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-r",
        str(REPO_DIR / "requirements-frozenlake-colab.txt"),
    ],
    check=True,
)

flash_status = Path("/content/frozenlake_flash_install_status.txt")
if PREFER_FLASH_ATTENTION:
    flash_env = os.environ.copy()
    flash_env["MAX_JOBS"] = "4"
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "flash-attn", "--no-build-isolation"],
        env=flash_env,
        check=False,
    )
    flash_status.write_text(str(result.returncode), encoding="utf-8")
    if result.returncode:
        print("FlashAttention installation failed; the notebook will use SDPA.")
else:
    flash_status.write_text("skipped", encoding="utf-8")

if importlib.util.find_spec("google.colab") is not None:
    print("Dependencies installed. Restarting the Colab Python process now.")
    os.kill(os.getpid(), signal.SIGKILL)
else:
    print("Dependencies installed. Restart the Python kernel manually, then continue below.")

## Continue here after the runtime reconnects

The configuration file under `/content` survives a normal Colab runtime restart. Run the next cell to restore all variables.

In [ ]:
# @title 5. Restore configuration after restart
from pathlib import Path
import importlib.util
import json
import os
import subprocess
import sys

CONFIG_PATH = Path("/content/frozenlake_lvr_colab_config.json")
if not CONFIG_PATH.is_file():
    raise RuntimeError("Configuration is missing. Run the Configuration cell first.")

config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
REPO_URL = config["repo_url"]
REVISION = config["revision"]
DRIVE_URL = config["drive_url"]
ARCHIVE_SHA256 = config["archive_sha256"].upper()
EXPECTED_ARCHIVE_BYTES = int(config["expected_archive_bytes"])
MODEL_NAME = config["model_name"]
PREFER_FLASH_ATTENTION = bool(config["prefer_flash_attention"])

REPO_DIR = Path("/content/lvr")
ARCHIVE_PATH = Path("/content/frozenlake_training_samples.zip")
DATA_ROOT = REPO_DIR / "training_samples" / "frozenlake"
if not (REPO_DIR / ".git").is_dir():
    raise RuntimeError("Repository clone is missing. Run the clone cell before installing.")

os.chdir(REPO_DIR)
has_flash_attention = importlib.util.find_spec("flash_attn") is not None
ATTENTION = "flash_attention_2" if PREFER_FLASH_ATTENTION and has_flash_attention else "sdpa"
DISABLE_FLASH_ATTN2 = "False" if ATTENTION == "flash_attention_2" else "True"

import torch
import transformers
if torch.__version__.split("+")[0] != "2.6.0":
    raise RuntimeError(f"Expected torch 2.6.0 after restart; found {torch.__version__}.")
print(f"Repository: {REPO_DIR}")
print(f"Torch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"Attention backend: {ATTENTION}")

In [ ]:
# @title 6. Download the FrozenLake ZIP and verify its exact checksum
import hashlib
from pathlib import Path
import gdown

def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest().upper()

if ARCHIVE_PATH.is_file():
    existing_hash = sha256_file(ARCHIVE_PATH)
    if ARCHIVE_PATH.stat().st_size != EXPECTED_ARCHIVE_BYTES or existing_hash != ARCHIVE_SHA256:
        raise RuntimeError(
            f"Existing archive failed verification: {ARCHIVE_PATH}. "
            "Move it aside and rerun this cell."
        )
    print("Verified existing archive; download skipped.")
else:
    partial_path = ARCHIVE_PATH.with_suffix(".zip.part")
    downloaded = gdown.download(
        url=DRIVE_URL,
        output=str(partial_path),
        quiet=False,
        fuzzy=True,
        resume=True,
    )
    if not downloaded or not partial_path.is_file():
        raise RuntimeError("Google Drive download did not produce the expected file.")

    actual_bytes = partial_path.stat().st_size
    actual_hash = sha256_file(partial_path)
    print(f"Downloaded bytes: {actual_bytes}")
    print(f"SHA-256: {actual_hash}")
    if actual_bytes != EXPECTED_ARCHIVE_BYTES:
        raise RuntimeError(f"Archive size mismatch: expected {EXPECTED_ARCHIVE_BYTES}, got {actual_bytes}.")
    if actual_hash != ARCHIVE_SHA256:
        raise RuntimeError(f"Archive checksum mismatch: expected {ARCHIVE_SHA256}, got {actual_hash}.")
    partial_path.replace(ARCHIVE_PATH)

print(f"Archive ready: {ARCHIVE_PATH} ({ARCHIVE_PATH.stat().st_size / 1024**3:.3f} GiB)")

In [ ]:
# @title 7. Safely extract into training_samples/frozenlake
import os
from pathlib import Path
from zipfile import ZipFile

EXPECTED_PNGS = 16806
EXPECTED_TRACES = 4000
data_parent = REPO_DIR / "training_samples"

def extracted_counts() -> tuple[int, int]:
    if not DATA_ROOT.is_dir():
        return 0, 0
    return (
        sum(1 for _ in DATA_ROOT.glob("[0-9]" * 8 + "/frame_*.png")),
        sum(1 for _ in DATA_ROOT.glob("[0-9]" * 8 + "/trace.json")),
    )

png_count, trace_count = extracted_counts()
if (png_count, trace_count) == (EXPECTED_PNGS, EXPECTED_TRACES):
    print("Dataset is already fully extracted; extraction skipped.")
else:
    data_parent.mkdir(parents=True, exist_ok=True)
    destination = data_parent.resolve()
    with ZipFile(ARCHIVE_PATH) as archive:
        infos = archive.infolist()
        files = [info.filename for info in infos if not info.is_dir()]
        png_entries = [
            name for name in files
            if name.startswith("frozenlake/") and name.endswith(".png")
        ]
        trace_entries = [
            name for name in files
            if name.startswith("frozenlake/") and name.endswith("/trace.json")
        ]
        unexpected = [
            name for name in files
            if not (
                name.startswith("frozenlake/")
                and (name.endswith(".png") or name.endswith("/trace.json"))
            )
        ]
        if len(png_entries) != EXPECTED_PNGS:
            raise RuntimeError(f"ZIP PNG count mismatch: {len(png_entries)}")
        if len(trace_entries) != EXPECTED_TRACES:
            raise RuntimeError(f"ZIP trace count mismatch: {len(trace_entries)}")
        if unexpected:
            raise RuntimeError(f"Unexpected ZIP entries: {unexpected[:10]}")

        for info in infos:
            target = (destination / info.filename).resolve()
            if target != destination and destination not in target.parents:
                raise RuntimeError(f"Unsafe path in ZIP: {info.filename}")
        archive.extractall(destination)

    png_count, trace_count = extracted_counts()

if (png_count, trace_count) != (EXPECTED_PNGS, EXPECTED_TRACES):
    raise RuntimeError(
        f"Extracted dataset mismatch: {png_count} PNGs, {trace_count} traces."
    )
print(f"Dataset ready: {DATA_ROOT}")
print(f"PNGs: {png_count}; traces: {trace_count}")

In [ ]:
# @title 8. Validate all manifests, paths, trajectories, and split isolation
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "scripts/validate_frozenlake_manifests.py",
        "--data-dir",
        "data/frozenlake",
        "--image-folder",
        str(DATA_ROOT),
    ],
    cwd=REPO_DIR,
    check=True,
)

In [ ]:
# @title 9. Run the Colab environment preflight
import json
import subprocess
import sys

preflight = subprocess.run(
    [
        sys.executable,
        "scripts/check_frozenlake_colab.py",
        "--attention",
        ATTENTION,
        "--workspace",
        str(REPO_DIR),
    ],
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
    check=False,
)
if preflight.stdout:
    print(preflight.stdout)
if preflight.stderr:
    print(preflight.stderr, file=sys.stderr)
if preflight.returncode:
    try:
        report = json.loads(preflight.stdout)
        errors = report.get("errors", [])
    except json.JSONDecodeError:
        errors = []
    detail = "\n".join(f"- {error}" for error in errors)
    if not detail:
        detail = "See the subprocess output immediately above."
    raise RuntimeError(f"FrozenLake preflight failed:\n{detail}")
print("FrozenLake preflight passed.")

In [ ]:
# @title 10. One-batch GPU forward smoke test
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "scripts/smoke_test_frozenlake_lvr.py",
        "--model",
        MODEL_NAME,
        "--data-path",
        "data/frozenlake/train.jsonl",
        "--image-folder",
        str(DATA_ROOT),
        "--attention",
        ATTENTION,
    ],
    cwd=REPO_DIR,
    check=True,
)
print("Do not continue unless the smoke test reported status: ok and finite losses.")

## Ten-step optimizer pilot

Enable this only after the one-batch smoke test succeeds. It tests DeepSpeed initialization, ZeRO-3 CPU offload, backward propagation, optimizer updates, and final model saving without writing a large intermediate optimizer checkpoint.

In [ ]:
# @title 11. Run the 10-step pilot
RUN_PILOT = False  # @param {type:"boolean"}
PILOT_OUTPUT_DIR = "/content/frozenlake_pilot"  # @param {type:"string"}

import os
from pathlib import Path
import subprocess

if not RUN_PILOT:
    print("Pilot disabled. Set RUN_PILOT=True after the smoke test passes.")
else:
    pilot_dir = Path(PILOT_OUTPUT_DIR)
    if pilot_dir.exists() and any(pilot_dir.iterdir()):
        raise RuntimeError(
            f"{pilot_dir} is not empty. Choose a new PILOT_OUTPUT_DIR to protect prior results."
        )
    env = os.environ.copy()
    env.update({
        "MODEL_NAME": MODEL_NAME,
        "DATA_PATH": "data/frozenlake/train.jsonl",
        "EVAL_DATA_PATH": "data/frozenlake/validation.jsonl",
        "IMAGE_FOLDER": str(DATA_ROOT),
        "OUTPUT_DIR": str(pilot_dir),
        "MAX_STEPS": "10",
        "SAVE_STEPS": "1000",
        "EVAL_STEPS": "1000",
        "DISABLE_FLASH_ATTN2": DISABLE_FLASH_ATTN2,
    })
    subprocess.run(
        ["bash", "scripts/finetune_lvr_frozenlake_3b_colab.sh"],
        cwd=REPO_DIR,
        env=env,
        check=True,
    )

## Full training

This cell mounts Google Drive and uses a persistent output directory. Direct Drive checkpoint writes can be slower than local disk, but the retained checkpoint survives a Colab runtime reset.

For a fresh run, leave `CHECKPOINT_NAME` empty and ensure the output directory is empty. To resume, set it to the exact `checkpoint-N` directory inside the same output directory.

In [ ]:
# @title 12. Run or resume full training
RUN_FULL_TRAINING = False  # @param {type:"boolean"}
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/lvr_frozenlake/qwen2.5-vl-3b"  # @param {type:"string"}
CHECKPOINT_NAME = ""  # @param {type:"string"}
NUM_TRAIN_EPOCHS = 5  # @param {type:"integer"}

import os
from pathlib import Path
import subprocess

if not RUN_FULL_TRAINING:
    print("Full training disabled. Enable it only after the 10-step pilot succeeds.")
else:
    from google.colab import drive
    drive.mount("/content/drive")

    output_dir = Path(DRIVE_OUTPUT_DIR)
    if output_dir.exists() and any(output_dir.iterdir()) and not CHECKPOINT_NAME:
        raise RuntimeError(
            f"{output_dir} is not empty. Set CHECKPOINT_NAME to resume or choose a new output directory."
        )
    output_dir.mkdir(parents=True, exist_ok=True)

    env = os.environ.copy()
    env.update({
        "MODEL_NAME": MODEL_NAME,
        "DATA_PATH": "data/frozenlake/train.jsonl",
        "EVAL_DATA_PATH": "data/frozenlake/validation.jsonl",
        "IMAGE_FOLDER": str(DATA_ROOT),
        "OUTPUT_DIR": str(output_dir),
        "CHECKPOINT_NAME": CHECKPOINT_NAME,
        "NUM_TRAIN_EPOCHS": str(NUM_TRAIN_EPOCHS),
        "MAX_STEPS": "-1",
        "DISABLE_FLASH_ATTN2": DISABLE_FLASH_ATTN2,
    })
    subprocess.run(
        ["bash", "scripts/finetune_lvr_frozenlake_3b_colab.sh"],
        cwd=REPO_DIR,
        env=env,
        check=True,
    )

In [ ]:
# @title 13. Inspect current GPU, RAM, disk, and checkpoints
import shutil
import subprocess
import psutil
from pathlib import Path

subprocess.run(["nvidia-smi"], check=True)
memory = psutil.virtual_memory()
disk = shutil.disk_usage("/content")
print(f"RAM used: {(memory.total - memory.available) / 1024**3:.1f} GiB")
print(f"RAM free: {memory.available / 1024**3:.1f} GiB")
print(f"Local disk free: {disk.free / 1024**3:.1f} GiB")

output_dir = Path(DRIVE_OUTPUT_DIR)
if output_dir.exists():
    checkpoints = sorted(
        output_dir.glob("checkpoint-*"),
        key=lambda path: int(path.name.split("-")[-1]),
    )
    print("Checkpoints:", [str(path) for path in checkpoints])

## Evaluation gates

Use validation while selecting the latent-end threshold. Run the final test split only once after the configuration is fixed.

In [ ]:
# @title 14. Validation preview
RUN_VALIDATION = False  # @param {type:"boolean"}
EVALUATION_CHECKPOINT = "/content/drive/MyDrive/lvr_frozenlake/qwen2.5-vl-3b"  # @param {type:"string"}
VALIDATION_MAX_SAMPLES = 20  # @param {type:"integer"}
LVR_END_THRESHOLD = 0.02  # @param {type:"number"}

import subprocess
import sys

if not RUN_VALIDATION:
    print("Validation disabled.")
else:
    command = [
        sys.executable,
        "evaluation/evaluate_frozenlake_lvr.py",
        "--checkpoint",
        EVALUATION_CHECKPOINT,
        "--data-path",
        "data/frozenlake/validation.jsonl",
        "--image-folder",
        str(DATA_ROOT),
        "--output-dir",
        str(Path(EVALUATION_CHECKPOINT) / "validation_evaluation"),
        "--lvr-end-threshold",
        str(LVR_END_THRESHOLD),
        "--attention",
        ATTENTION,
    ]
    if VALIDATION_MAX_SAMPLES > 0:
        command.extend(["--max-samples", str(VALIDATION_MAX_SAMPLES)])
    subprocess.run(command, cwd=REPO_DIR, check=True)

In [ ]:
# @title 15. Final test evaluation
RUN_FINAL_TEST = False  # @param {type:"boolean"}

import subprocess
import sys
from pathlib import Path

if not RUN_FINAL_TEST:
    print("Final test disabled. Enable it only after threshold selection is finished.")
else:
    subprocess.run(
        [
            sys.executable,
            "evaluation/evaluate_frozenlake_lvr.py",
            "--checkpoint",
            EVALUATION_CHECKPOINT,
            "--data-path",
            "data/frozenlake/test.jsonl",
            "--image-folder",
            str(DATA_ROOT),
            "--output-dir",
            str(Path(EVALUATION_CHECKPOINT) / "test_evaluation"),
            "--lvr-end-threshold",
            str(LVR_END_THRESHOLD),
            "--attention",
            ATTENTION,
        ],
        cwd=REPO_DIR,
        check=True,
    )

## Expected data invariants

A successful dataset validation reports:

- 4,000 total samples: 3,600 train, 200 validation, 200 test
- 12,806 transitions
- 16,806 referenced images
- zero board-layout overlap between splits

A successful smoke test reports `status: ok`, matching latent/target token counts, and finite language, latent, and mode-switch losses.